# Exploring Atlanta's flat street network

This notebook is for **exploration only**. The analysis itself runs from
the CLI (`python -m sf_flat_routes all`) and every figure in the README
and in `outputs/findings.md` is produced there, not here.

It assumes the pipeline has already been run, so the cached products in
`data/processed/` exist.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt

from sf_flat_routes.utils import setup_logging, configure_gdal_for_proxy
from sf_flat_routes.pipeline import build_context
from sf_flat_routes.config import ROUTING_PROFILES

setup_logging(); configure_gdal_for_proxy()
ctx = build_context()
edges, directed = ctx.edges, ctx.directed
print(f'{len(edges):,} edges, {len(directed):,} directed arcs')


## How steep is Atlanta, really?


In [ ]:
w = edges[edges.walk_ok]
km = w.length_m.sum()/1000
print(f'walkable network: {km:,.0f} km, {w.cum_gain_fwd.sum()/km:.1f} m climb per km')

bins=[0,.03,.05,.08,.10,.15,.62]
share=[w[(w.max_abs_grade>=lo)&(w.max_abs_grade<hi)].length_m.sum()/w.length_m.sum()
       for lo,hi in zip(bins[:-1],bins[1:])]
fig,ax=plt.subplots(figsize=(7,3.5))
ax.bar([f'{lo:.0%}-{hi:.0%}' for lo,hi in zip(bins[:-1],bins[1:])],
       [100*s for s in share], color='#2c7bb6')
ax.set_ylabel('% of street length'); ax.set_xlabel('maximum gradient')
ax.set_title('Distribution of street gradient in Atlanta')
plt.tight_layout()


## Elevation profile of a single street

Pick any street and look at what the lidar says about it.


In [ ]:
from sf_flat_routes.elevation import PROFILES_NPZ

npz = np.load(PROFILES_NPZ)
pos = {int(e): i for i, e in enumerate(npz['edge_ids'])}
off, D, E = npz['offsets'], npz['dist'], npz['elev']

def profile(edge_id):
    i = pos[int(edge_id)]
    return D[off[i]:off[i+1]], E[off[i]:off[i+1]]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for ax, name in zip(axes, ['Mary George Avenue Northwest', 'Atlanta Beltline Eastside Trail']):
    sub = edges[edges.name == name].nlargest(1, 'max_abs_grade')
    d, z = profile(sub.edge_id.iloc[0])
    ax.plot(d, z, lw=2, color='#08306b')
    ax.set_title(f'{name}  (max {sub.max_abs_grade.iloc[0]:.1%})')
    ax.set_xlabel('metres along edge'); ax.set_ylabel('elevation (m)')
plt.tight_layout()


## Compare routing objectives for one trip


In [ ]:
from sf_flat_routes.routing import route, route_profile

graph = ctx.graphs['walk']
pts = dict(zip(ctx.points['walk'].neighborhood, ctx.points['walk'].node))
o, d = 'Midtown', 'Grant Park'

fig, ax = plt.subplots(figsize=(10, 4))
for pname, colour in zip(['shortest','balanced','min_climb'],
                         ['#e3492e','#fdae61','#2c7bb6']):
    arcs, s = route(graph, pts[o], pts[d], ROUTING_PROFILES[pname])
    dist, elev = route_profile(graph, arcs, ctx.profiles)
    ax.plot(dist/1000, elev, label=f"{pname}: {s['distance_m']/1609:.2f} mi, "
            f"{s['elev_gain_m']*3.28084:.0f} ft climb", color=colour, lw=2)
ax.set_xlabel('km along route'); ax.set_ylabel('elevation (m)')
ax.set_title(f'{o} to {d}: three definitions of "flat"')
ax.legend(); plt.tight_layout()


## The Pareto frontier for a pair

Sweep the climbing weight and watch distance trade against ascent.


In [ ]:
from sf_flat_routes.pairs import pareto_for_pairs

pf = pareto_for_pairs(graph, ctx.points['walk'], [('Midtown','Grant Park')])
pf = pf[pf.pareto_optimal].sort_values('distance_m')

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(pf.distance_m/1609.344, pf.elev_gain_m*3.28084, 'o-', color='#08306b')
for _, r in pf.iterrows():
    ax.annotate(f"a={r.alpha:g}", (r.distance_m/1609.344, r.elev_gain_m*3.28084),
                textcoords='offset points', xytext=(6,4), fontsize=8)
ax.set_xlabel('distance (miles)'); ax.set_ylabel('climbing (feet)')
ax.set_title('Midtown to Grant Park: distance vs climbing frontier')
plt.tight_layout()


## Where are the corridors and the passes?


In [ ]:
cor = gpd.read_file('../outputs/flat_corridors.gpkg')
pz  = gpd.read_file('../outputs/passes.geojson').to_crs(edges.crs)

fig, ax = plt.subplots(figsize=(9, 10))
ctx.neighborhoods.plot(ax=ax, facecolor='#f6f5f2', edgecolor='#d5dade', lw=.6)
edges[edges.walk_ok].plot(ax=ax, color='#b8c0c8', lw=.15)
cor[cor['mode']=='walk'].plot(ax=ax, color='#2c7bb6', lw=2.4)
mid = pz.geometry.interpolate(.5, normalized=True)
ax.scatter([p.x for p in mid], [p.y for p in mid], s=40, marker='^',
           facecolor='#ffd166', edgecolor='#4a3b10', zorder=5)
ax.set_aspect('equal'); ax.set_axis_off()
ax.set_title('Flat corridors (blue) and critical passes (triangles)')
plt.tight_layout()


## Interrogate the pair matrix


In [ ]:
pairs = pd.read_parquet('../data/processed/neighborhood_pairs.parquet')
mc = pairs[(pairs['mode']=='walk') & (pairs.profile=='min_climb')]
cols = ['origin','destination','shortest_gain_m','elev_gain_m',
        'gain_saved_pct','detour_ratio']
mc.nlargest(12, 'gain_saved_pct')[cols].round(2)
